# BCD V_DIFF — Bad Change Detection Demo

This notebook demonstrates both BCD detectors on synthetic time-series data.

- **V_DIFF (v1)** — DBSCAN-based: slices the old-build history into windows and treats the new-build window as a candidate outlier.
- **V_DIFF_V2** — Gate-based: five sequential statistical checks against a long baseline (floor → percentile rank → spike suppression → correlated check → robust z-score).

Each scenario uses a synthetic old-build baseline and a new-build series that either regresses or stays healthy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from anomaly_detection.bcd import V_DIFF, V_DIFF_V2

rng = np.random.default_rng(42)

## Synthetic data helpers

In [ ]:
def make_old_build(n_days=7, pts_per_day=1440, base=50.0, noise=5.0, seed=42):
    """Stable old-build baseline: ~7 days of minute-level data."""
    rng = np.random.default_rng(seed)
    n = n_days * pts_per_day
    t0 = int(pd.Timestamp('2024-01-01').timestamp())
    timestamps = np.arange(t0, t0 + n * 60, 60)
    values = base + rng.normal(0, noise, n).clip(-noise * 2, noise * 2)
    return pd.Series(values, index=timestamps)


def make_new_build_regression(n_pts=30, base=50.0, spike=200.0, noise=5.0, seed=1):
    """New-build series with a clear regression: values jump to ~spike."""
    rng = np.random.default_rng(seed)
    t0 = int(pd.Timestamp('2024-01-08').timestamp())
    timestamps = np.arange(t0, t0 + n_pts * 60, 60)
    values = spike + rng.normal(0, noise, n_pts)
    return pd.Series(values, index=timestamps)


def make_new_build_healthy(n_pts=30, base=50.0, noise=5.0, seed=2):
    """New-build series with no regression: same level as old build."""
    rng = np.random.default_rng(seed)
    t0 = int(pd.Timestamp('2024-01-08').timestamp())
    timestamps = np.arange(t0, t0 + n_pts * 60, 60)
    values = base + rng.normal(0, noise, n_pts)
    return pd.Series(values, index=timestamps)


def plot_scenario(baseline_ts, new_ts, title, detected):
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(baseline_ts.values[-200:], color='steelblue', alpha=0.6, label='Old build (tail)')
    ax.plot(
        range(200, 200 + len(new_ts)), new_ts.values,
        color='crimson' if detected else 'seagreen',
        linewidth=2,
        label=f'New build — {"ANOMALY DETECTED" if detected else "No anomaly"}',
    )
    if detected:
        threshold = np.mean(baseline_ts.values) + 3 * np.std(baseline_ts.values)
        anomaly_idx = np.where(new_ts.values > threshold)[0][:3]
        if len(anomaly_idx) > 0:
            ax.scatter(
                [200 + i for i in anomaly_idx],
                new_ts.values[anomaly_idx],
                color='red', zorder=5, s=80,
                label=f'First {len(anomaly_idx)} anomalous point{"s" if len(anomaly_idx) > 1 else ""}',
            )
    ax.axvline(200, color='gray', linestyle='--', linewidth=1)
    ax.set_title(title)
    ax.legend(loc='upper left')
    ax.set_xlabel('Time (minutes)')
    ax.set_ylabel('Metric value')
    plt.tight_layout()
    plt.show()
    print(f'  Result: {"ANOMALY DETECTED" if detected else "No anomaly"}')

---
## V_DIFF (v1) — DBSCAN-based detector

In [3]:
detector_v1 = V_DIFF(eps=0.3, minPts=2, avgMin=20.0, slopeMin=-30.0, giniMax=0.3, win=600)
print('V_DIFF parameters:', vars(detector_v1))

V_DIFF parameters: {'eps': 0.3, 'minPts': 2, 'avgMin': 20.0, 'slopeMin': -30.0, 'giniMax': 0.3, 'win': 600}


### Scenario 1 — Regression (expect: detected)

In [ ]:
baseline_ts = make_old_build(n_days=7, base=50.0, noise=4.0)
new_ts = make_new_build_regression(n_pts=30, spike=200.0, noise=5.0)

result = detector_v1.detect(baseline_ts.copy(), new_ts.copy())
plot_scenario(baseline_ts, new_ts, 'V_DIFF v1 — Regression scenario', result)

### Scenario 2 — Healthy deploy (expect: no anomaly)

In [ ]:
baseline_ts = make_old_build(n_days=7, base=50.0, noise=4.0)
new_ts = make_new_build_healthy(n_pts=30, base=50.0, noise=4.0)

result = detector_v1.detect(baseline_ts.copy(), new_ts.copy())
plot_scenario(baseline_ts, new_ts, 'V_DIFF v1 — Healthy deploy scenario', result)

### Scenario 3 — Correlated event: both builds spike (expect: detected)

V_DIFF v1 has no correlated check — it only compares the new-build window against old-build windows. When both builds are elevated simultaneously, it still flags the new build.

In [ ]:
rng_event = np.random.default_rng(5)
t_new = int(pd.Timestamp('2024-01-08').timestamp())

baseline_ts_base = make_old_build(n_days=7, base=50.0, noise=4.0)
new_vals = np.concatenate([50.0 + rng_event.normal(0, 4, 15), 120.0 + rng_event.normal(0, 5, 15)])
new_ts = pd.Series(new_vals, index=np.arange(t_new, t_new + 30 * 60, 60))
current_time = new_ts.index[-1]
event_idx = np.arange(current_time - 9 * 60, current_time + 60, 60)
concurrent_old = pd.Series(100.0 + rng_event.normal(0, 5, len(event_idx)), index=event_idx)
baseline_ts_with_event = pd.concat([baseline_ts_base, concurrent_old]).sort_index()

result = detector_v1.detect(baseline_ts_with_event.copy(), new_ts.copy())
plot_scenario(baseline_ts_with_event, new_ts, 'V_DIFF v1 — Correlated event', result)

### Scenario 4 — Sparse-spike baseline (expect: no anomaly)

The old build is silent most of the time with periodic bursts (high GINI). The new build has a similar burst. V_DIFF v1 does not detect it — DBSCAN clusters the new-build window with elevated old-build spike windows, so it is not labelled an outlier.

In [ ]:
rng_s = np.random.default_rng(7)
t0_s = int(pd.Timestamp('2024-01-01').timestamp())
t_new = int(pd.Timestamp('2024-01-08').timestamp())
n = 10080

vals = np.zeros(n)
vals[rng_s.choice(n, size=int(n * 0.30), replace=False)] = 150.0 + rng_s.normal(0, 2, int(n * 0.30))
baseline_ts_spiky = pd.Series(vals, index=np.arange(t0_s, t0_s + n * 60, 60))

new_vals = np.zeros(30)
new_vals[-5:] = 165.0 + np.random.default_rng(3).normal(0, 2, 5)
new_ts_spike = pd.Series(new_vals, index=np.arange(t_new, t_new + 30 * 60, 60))

result = detector_v1.detect(baseline_ts_spiky.copy(), new_ts_spike.copy())
plot_scenario(baseline_ts_spiky, new_ts_spike, 'V_DIFF v1 — Sparse-spike baseline', result)

---
## V_DIFF_V2 — Gate-based detector

In [8]:
detector_v2 = V_DIFF_V2(
    threshold=10.0,
    avg_min=10.0,
    correlated_check=True,
    adaptive_pct=True,
    suppress_similar_spikes=True,
)
print('V_DIFF_V2 parameters:', vars(detector_v2))

V_DIFF_V2 parameters: {'threshold': 10.0, 'avg_min': 10.0, 'correlated_check': True, 'adaptive_pct': True, 'suppress_similar_spikes': True, 'spike_range_multiplier': 1.2, 'baseline_spike_calculation_method': 'top5_median', 'baseline_spikiness_gini_threshold': 0.4, 'baseline_variance_cv_threshold': 0.45, 'correlation_ratio_threshold': 2.0, 'concurrent_spike_min': 50.0, 'min_baseline_coverage': 0.1, 'floor_window_high': 5, 'floor_window_low': 3, 'pct_gate_high_cv': 0.999, 'pct_gate_low_cv': 0.99, 'cv_high_threshold': 0.8, 'min_gini_observations': 10}


### Scenario 1 — Regression (expect: detected)

In [ ]:
baseline_ts = make_old_build(n_days=7, base=50.0, noise=4.0)
new_ts = make_new_build_regression(n_pts=30, spike=200.0, noise=5.0)

result = detector_v2.detect(baseline_ts.copy(), new_ts.copy())
plot_scenario(baseline_ts, new_ts, 'V_DIFF_V2 — Regression scenario', result)

### Scenario 2 — Healthy deploy (expect: no anomaly)

In [ ]:
baseline_ts = make_old_build(n_days=7, base=50.0, noise=4.0)
new_ts = make_new_build_healthy(n_pts=30, base=50.0, noise=4.0)

result = detector_v2.detect(baseline_ts.copy(), new_ts.copy())
plot_scenario(baseline_ts, new_ts, 'V_DIFF_V2 — Healthy deploy scenario', result)

### Scenario 3 — Correlated event: both builds spike (expect: suppressed)

Same data as V1 scenario 3. V_DIFF_V2's correlated check compares the new-build median against the concurrent old-build median — when the ratio is below the threshold, both builds are considered co-elevated and the alert is suppressed.

In [ ]:
rng_event = np.random.default_rng(5)
t_new = int(pd.Timestamp('2024-01-08').timestamp())

baseline_ts_base = make_old_build(n_days=7, base=50.0, noise=4.0)
new_vals = np.concatenate([50.0 + rng_event.normal(0, 4, 15), 120.0 + rng_event.normal(0, 5, 15)])
new_ts = pd.Series(new_vals, index=np.arange(t_new, t_new + 30 * 60, 60))
current_time = new_ts.index[-1]
event_idx = np.arange(current_time - 9 * 60, current_time + 60, 60)
concurrent_old = pd.Series(100.0 + rng_event.normal(0, 5, len(event_idx)), index=event_idx)
baseline_ts_with_event = pd.concat([baseline_ts_base, concurrent_old]).sort_index()

result = detector_v2.detect(baseline_ts_with_event.copy(), new_ts.copy())
plot_scenario(baseline_ts_with_event, new_ts, 'V_DIFF_V2 — Correlated event (both builds spike, expect suppression)', result)

### Scenario 4 — Sparse-spike baseline (expect: suppressed)

Same data as V1 scenario 4. V_DIFF_V2 explicitly computes the baseline GINI — it is high (sparse metric), the new burst is within the historical spike ceiling, so the spike suppression gate fires.

In [ ]:
rng_s = np.random.default_rng(7)
t0_s = int(pd.Timestamp('2024-01-01').timestamp())
t_new = int(pd.Timestamp('2024-01-08').timestamp())
n = 10080

vals = np.zeros(n)
vals[rng_s.choice(n, size=int(n * 0.30), replace=False)] = 150.0 + rng_s.normal(0, 2, int(n * 0.30))
baseline_ts_spiky = pd.Series(vals, index=np.arange(t0_s, t0_s + n * 60, 60))

new_vals = np.zeros(30)
new_vals[-5:] = 165.0 + np.random.default_rng(3).normal(0, 2, 5)
new_ts_spike = pd.Series(new_vals, index=np.arange(t_new, t_new + 30 * 60, 60))

result = detector_v2.detect(baseline_ts_spiky.copy(), new_ts_spike.copy())
plot_scenario(baseline_ts_spiky, new_ts_spike, 'V_DIFF_V2 — Sparse-spike baseline (spike suppression gate)', result)

---
## Summary

All four scenarios use identical data for both detectors.

| Scenario | V_DIFF v1 | V_DIFF_V2 |
|---|---|---|
| Regression | Detected | Detected |
| Healthy deploy | No anomaly | No anomaly |
| Correlated event | **Detected** | **Suppressed** |
| Sparse-spike baseline | No anomaly | No anomaly |

**Scenarios 1 & 2** — both algorithms agree. A clear regression is detected; a healthy deploy is not.

**Scenario 3 (Correlated event)** — the key contrast. V_DIFF v1 has no awareness of what the old build is doing concurrently, so it flags the new build. V_DIFF_V2's correlated check sees that both builds are elevated together and suppresses.

**Scenario 4 (Sparse-spike baseline)** — both produce no anomaly, but for different reasons. V_DIFF v1: DBSCAN clusters the new-build window with elevated old-build spike windows, so it is not marked an outlier. V_DIFF_V2: the spike suppression gate explicitly computes the baseline GINI, determines the metric is naturally spiky, and suppresses because the new burst is within the historical spike ceiling.